# Day 18 – Sampling Techniques
## Making Reliable Conclusions from Large Datasets

**30 Days of Data Analytics – From Foundations to Decision Intelligence**

Sampling is not simply taking fewer rows. It is the disciplined process of selecting a subset of a population so useful conclusions can be drawn about the larger population.

**Dataset:** NYC 311 Service Requests  
**Tools:** Python, Pandas, Matplotlib

## Learning Objectives

- Understand population vs. sample
- Perform simple random sampling
- Perform systematic sampling
- Perform stratified sampling
- Compare sample and population distributions
- Compare population and sample statistics
- Demonstrate sampling variability
- Demonstrate sampling bias
- Understand how to choose an appropriate sampling method

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## 1. Load the Dataset

Place `nyc_311_sample.csv` in the same directory as this notebook, or change `DATA_PATH` below.

In [ ]:
DATA_PATH = "nyc_311_sample.csv"
df = pd.read_csv(DATA_PATH)

print("Number of records:", len(df))
df.head()

## 2. Understand the Population

The complete set of records available for the analysis is the **population**. A sample is a subset selected from this population.

In [ ]:
print("Population size:", len(df))
print("Columns:", list(df.columns))

## 3. Simple Random Sampling

Each record has an equal opportunity to be selected. `random_state` makes the result reproducible.

In [ ]:
sample_size = min(10000, len(df))

random_sample = df.sample(
    n=sample_size,
    random_state=42
)

print("Population:", len(df))
print("Random sample:", len(random_sample))

## 4. Sampling by Percentage

In [ ]:
sample_10 = df.sample(frac=0.10, random_state=42)

print("Population:", len(df))
print("10% sample:", len(sample_10))

## 5. Compare Population and Sample by Borough

Before generalizing results, check whether important population characteristics are represented in the sample.

In [ ]:
if "Borough" in df.columns:
    population_borough = df["Borough"].value_counts(normalize=True, dropna=False).mul(100)
    sample_borough = sample_10["Borough"].value_counts(normalize=True, dropna=False).mul(100)

    comparison = pd.DataFrame({
        "Population %": population_borough,
        "Sample %": sample_borough
    }).fillna(0)

    display(comparison)
else:
    print("Column 'Borough' was not found.")

## 6. Visualize Population vs Sample

In [ ]:
if "Borough" in df.columns:
    comparison.plot(kind="bar", figsize=(10, 5))
    plt.title("Population vs Sample – Borough Distribution")
    plt.ylabel("Percentage")
    plt.xlabel("Borough")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. Systematic Sampling

Systematic sampling selects observations at regular intervals. A randomized starting position is used here.

In [ ]:
interval = 10
start = np.random.default_rng(42).integers(0, interval)

systematic_sample = df.iloc[start::interval]

print("Starting position:", start)
print("Population:", len(df))
print("Systematic sample:", len(systematic_sample))

## 8. Stratified Sampling

Stratified sampling divides the population into meaningful groups and samples within each group. Borough is used as an example.

In [ ]:
if "Borough" in df.columns:
    stratified_sample = (
        df.groupby("Borough", group_keys=False, dropna=False)
          .apply(lambda x: x.sample(frac=0.10, random_state=42))
          .reset_index(drop=True)
    )
    print("Population:", len(df))
    print("Stratified sample:", len(stratified_sample))
else:
    stratified_sample = pd.DataFrame()
    print("Borough column not available.")

## 9. Compare Sampling Methods

In [ ]:
if "Borough" in df.columns:
    random_distribution = sample_10["Borough"].value_counts(normalize=True, dropna=False).mul(100)
    systematic_distribution = systematic_sample["Borough"].value_counts(normalize=True, dropna=False).mul(100)
    stratified_distribution = stratified_sample["Borough"].value_counts(normalize=True, dropna=False).mul(100)

    sampling_comparison = pd.DataFrame({
        "Population %": population_borough,
        "Random %": random_distribution,
        "Systematic %": systematic_distribution,
        "Stratified %": stratified_distribution
    }).fillna(0)

    display(sampling_comparison)

## 10. Prepare Resolution Time

If Created Date and Closed Date are available, calculate resolution time and compare population and sample estimates.

In [ ]:
created_col = "Created Date"
closed_col = "Closed Date"

if created_col in df.columns and closed_col in df.columns:
    df[created_col] = pd.to_datetime(df[created_col], errors="coerce")
    df[closed_col] = pd.to_datetime(df[closed_col], errors="coerce")

    df["Resolution Hours"] = (
        df[closed_col] - df[created_col]
    ).dt.total_seconds() / 3600

    print(df["Resolution Hours"].describe())
else:
    print("Created Date and/or Closed Date columns were not found.")

## 11. Population Mean vs Sample Mean

In [ ]:
if "Resolution Hours" in df.columns:
    population_mean = df["Resolution Hours"].mean()
    sample_mean = sample_10["Resolution Hours"].mean()

    print("Population mean:", population_mean)
    print("Sample mean:", sample_mean)

## 12. Sampling Error

In [ ]:
if "Resolution Hours" in df.columns:
    sampling_error = sample_mean - population_mean

    print("Population mean:", population_mean)
    print("Sample mean:", sample_mean)
    print("Sampling error:", sampling_error)

## 13. Repeated Sampling and Sampling Variability

In [ ]:
if "Resolution Hours" in df.columns:
    sample_means = []

    for seed in range(100):
        sample = df.sample(frac=0.10, random_state=seed)
        sample_means.append(sample["Resolution Hours"].mean())

    sample_means = pd.Series(sample_means, name="Sample Mean")
    display(sample_means.head())

## 14. Distribution of Sample Means

In [ ]:
if "Resolution Hours" in df.columns:
    plt.figure(figsize=(10, 5))
    plt.hist(sample_means.dropna(), bins=20)
    plt.axvline(population_mean, linestyle="--", linewidth=2, label="Population Mean")
    plt.xlabel("Sample Mean")
    plt.ylabel("Frequency")
    plt.title("Distribution of Sample Means")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("Standard deviation of sample means:", sample_means.std())

## 15. Demonstrating Sampling Bias

Selecting records from only one borough is deliberately biased if the goal is to understand the entire city. This illustrates why a large sample is not automatically representative.

In [ ]:
if "Borough" in df.columns:
    first_borough = df["Borough"].dropna().iloc[0]
    biased_sample = df[df["Borough"] == first_borough]

    print("Selected Borough:", first_borough)
    print("Biased sample size:", len(biased_sample))

    if "Resolution Hours" in df.columns:
        print("Population mean:", df["Resolution Hours"].mean())
        print("Biased sample mean:", biased_sample["Resolution Hours"].mean())

## 16. Convenience Sampling

Selecting the first records is convenient but can introduce time or ordering bias. This is a teaching example, not a recommended population-inference method.

In [ ]:
convenience_sample = df.head(min(10000, len(df)))

print("Convenience sample size:", len(convenience_sample))

if "Resolution Hours" in df.columns:
    print("Population mean:", df["Resolution Hours"].mean())
    print("Convenience sample mean:", convenience_sample["Resolution Hours"].mean())

## 17. Compare Different Sample Sizes

Increasing sample size generally reduces sampling variability, but it does not correct systematic bias.

In [ ]:
if "Resolution Hours" in df.columns:
    fractions = [0.01, 0.05, 0.10, 0.20]
    results = []

    for fraction in fractions:
        sample = df.sample(frac=fraction, random_state=42)
        results.append({
            "Sample %": fraction * 100,
            "Sample Size": len(sample),
            "Mean Resolution Hours": sample["Resolution Hours"].mean()
        })

    sample_size_comparison = pd.DataFrame(results)
    display(sample_size_comparison)

# Key Takeaways

- **Population** = the complete set we want to understand.
- **Sample** = a subset selected from that population.
- **Simple random sampling** gives records an equal selection opportunity.
- **Systematic sampling** selects records at regular intervals.
- **Stratified sampling** helps ensure important subgroups are represented.
- **Sampling error** is expected because a sample is only part of the population.
- **Sampling bias** can systematically distort conclusions.
- Sample distributions should be compared with population distributions before generalizing results.
- Sampling should be chosen according to the **analytical question and population structure**.

> **A good sample doesn't simply contain fewer records. It preserves the story of the population.**